In [2]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 4.6 MB/s eta 0:00:00a 0:00:01


In [3]:
import time
import json
import psycopg2
from kafka import KafkaProducer
from datetime import datetime

In [5]:
conn = psycopg2.connect(
    dbname="stream_db",
    user="airflow",
    password="airflow",
    host="postgres",
    port=5432
)
cur = conn.cursor()

In [ ]:
producer = KafkaProducer(
    bootstrap_servers="kafka:29092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

cur = conn.cursor()
last_id = 0

while True:
    cur.execute(
        """
        SELECT
            id,
            crypto_id,
            price_usd,
            market_cap,
            volume_24h,
            return_24h,
            volatility_proxy,
            anomaly_flag,
            fetched_at
        FROM crypto.fact_crypto_price
        WHERE id > %s
        ORDER BY id
        """,
        (last_id,)
    )

    rows = cur.fetchall()

    for r in rows:
        last_id = r[0]

        producer.send("coingecko", {
            "crypto_id": r[1],
            "price_usd": float(r[2]),
            "market_cap": float(r[3]),
            "volume_24h": float(r[4]),
            "return_24h": float(r[5]),
            "volatility_proxy": float(r[6]),
            "anomaly_flag": int(r[7]),
            "time": str(r[8])
        })

    producer.flush()
    time.sleep(3)